In [1]:
import pandas as pd
import numpy as np

# 1, Identification of business problem
    =>  Main goal: predict which order is prone to fraud or late delivery
    =>  Secondary goal: deploy Machine Learning models in Python directly into tableau using TabPy Library


 
# 2, Identification of data
    =>  This step involves data identification, initial requirements, collection
        quality and data insights
    =>  How large is the dataset do I need to collect?
    =>  Data quality?
    =>  Is there any requirement, restriction related to the data collection?




# 3, Collect and prepare the data
    Data Investigation: How do I collect the data? 
    =>  The data will be retrieved from Mendeley Data, DataCo Global supply chain
    =>  Url: https://data.mendeley.com/datasets/8gx2fvg2k6/5
    Preprocess the raw data:
    =>  Elimination (LDA)
    =>  Normalization (StandardScaler, MinMaxScaler)
    =>  Removal of duplicate
    =>  Removal of outlier
    Data visualization:
    =>  PCA
    =>  Seaborn


In [2]:
data = pd.read_csv("DataCo Global - Supply Chain Company/DataCoSupplyChainDataset.csv")
# Any customer that did not provide zipcode will be assigned to 0 as default zipcode
data["Customer Zipcode"] = data["Customer Zipcode"].fillna(0)
print("Column names: \n{}".format(data.columns))
# print(data.head())
# print(data.isnull().sum())
print("Data size: {}".format(data.shape))

Column names: 
Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id',
       'Customer Lname', 'Customer Password', 'Customer Segment',
       'Customer State', 'Customer Street', 'Customer Zipcode',
       'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market',
       'Order City', 'Order Country', 'Order Customer Id',
       'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id',
       'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity', 'Sales', 'Order Item Total',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Order Zipcode', 'Product Card Id', 'Product Category Id',
   

    The feature "Customer Country" and "Order Country" are catergorical variable
    They need to be encoded to numerical variable

    We apply One-Hot Encoding when:
    => The categorical feature is not ordinal (like the countries above)
    => The number of categorical features is less so one-hot encoding can be effectively applied
    
    We apply Label Encoding when:
    => The categorical feature is ordinal (like Jr. kg, Sr. kg, Primary school, high school)
    => The number of categories is quite large as one-hot encoding can lead to high memory consumption
    
    ref: https://www.analyticsvidhya.com/blog/2020/03/one-hot-encoding-vs-label-encoding-using-scikit-learn/

In [3]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
data["Customer Country"] = label_encoder.fit_transform(data["Customer Country"])
data["Order Country"] = label_encoder.fit_transform(data["Order Country"])

    What factors determine an order is fraud?
    => Days for shipping (real), Days for shipment (scheduled), Customer Country

In [4]:
fraud_features = np.array(np.transpose([data["Days for shipping (real)"], data["Days for shipment (scheduled)"], data["Customer Country"]]))
fraud_labels = np.where(data["Order Status"] == 'SUSPECTED_FRAUD', 1, 0)

    
    What factors determine an order going to be late?
    => Days for shipment (scheduled), Order Country



In [5]:
late_features = np.transpose([data["Days for shipment (scheduled)"], data["Order Country"]])
late_labels = data["Late_delivery_risk"]


# 4, Choose and Train ML model
    Split data: 
    => For small dataset [0-10000] => K-fold cross validation: divide the entire data into k-fold evenly. 
        One fold will be used for test set and the rest will training set. 
    => For big dataset [10000 - inf] => shuffle the data first then split the dataset to 60% training set, 
        20% validation set, 20% test set
    Pick a model for supervised learning:
    => Logistic Regression
    => Naive Bayes
    => KNN
    => Decision Tree
    => Random Forest
    => SVM


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

training_fraud_features, test_fraud_features, training_fraud_class, test_fraud_class = train_test_split(fraud_features, fraud_labels, stratify=fraud_labels, train_size=0.7, test_size=0.3)
random_forest = RandomForestClassifier().fit(training_fraud_features, training_fraud_class)
fraud_model_accuracy = random_forest.score(test_fraud_features, test_fraud_class)
print(f"Fraud model accuracy: {fraud_model_accuracy}")

Fraud model accuracy: 0.9774909520644065


In [7]:
training_late_features, test_late_features, training_late_class, test_late_class = train_test_split(late_features, late_labels, stratify=late_labels, train_size=0.7, test_size=0.3)
random_forest = RandomForestClassifier().fit(training_late_features, training_late_class)
late_model_accuracy = random_forest.score(test_late_features, test_late_class)
print(f"Late model accuracy: {late_model_accuracy}")

Late model accuracy: 0.6962109461555507


In [9]:
data.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class




# 5, Evaluation
    => How well is my model doing? Is it a useful model?
    => Will training my model on more data improve its performance?
    => Do I need to include more features?
    => Do we care more about precision? or recall?
    => Accuracy itself alone will not be informative regarding to the model efficient
        


# 6, Tuning Parameters 
    => High bias => train longer 
    => High variant => need different models, more data



# 7, Interference or Prediction
    it’s time to utilize machine learning models in real-life scenarios. Congratulation !!!

